# Credit Risk & Loan Default Prediction
**Python | SQL-style analysis | Statistics | Machine Learning | Power BI**

## Project objective
Develop a model that estimates whether a loan will be charged off/defaulted, compare classification algorithms, examine borrower risk drivers, and calculate illustrative credit-risk indicators.

**Dataset option used here:** Lending Club loan data (`accepted_2007_to_2018Q4.csv`, or a similarly structured accepted-loans CSV).

> Dataset versions differ. This notebook checks for common Lending Club columns and selects only fields that exist. Review the dataset's data dictionary before interpreting fields. This is an educational project, not a lending or regulatory model.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost import unavailable:", e)

## 2. Load the Lending Club dataset
Attach your CSV using Kaggle **Add Input**. For a smaller first run, use a sampled or smaller Lending Club file. Large historical files may require substantial memory.

In [ ]:
DATA_PATH = None  # Example: "/kaggle/input/lending-club-loan-data/accepted_2007_to_2018Q4.csv"

if DATA_PATH is None:
    candidates = []
    if Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob("accepted_2007_to_2018Q4.csv"))
        if not candidates:
            candidates = [p for p in Path("/kaggle/input").rglob("*.csv")
                          if "loan" in p.name.lower() or "lending" in str(p).lower()]
    if not candidates:
        raise FileNotFoundError(
            "No Lending Club CSV found. Add a Lending Club dataset through Add Input "
            "or set DATA_PATH manually."
        )
    DATA_PATH = str(candidates[0])

# Read a manageable subset of columns first, then validate availability.
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Define the target variable
For Lending Club, `loan_status` commonly contains statuses such as Fully Paid, Charged Off, Current, Late, and In Grace Period.

For a clean historical binary target:
- Fully Paid = 0 (non-default)
- Charged Off = 1 (default proxy)

Exclude Current / late / in-progress loans because their final outcome is not yet known. Do not use post-origination fields that reveal the eventual outcome.

In [ ]:
if "loan_status" not in df.columns:
    raise ValueError("The selected CSV must contain a 'loan_status' column.")

print(df["loan_status"].value_counts(dropna=False).head(20))

status_map = {"Fully Paid": 0, "Charged Off": 1}
df = df[df["loan_status"].isin(status_map)].copy()
df["Default"] = df["loan_status"].map(status_map).astype(int)

print("Modeling shape:", df.shape)
print("Default rate:", round(df["Default"].mean()*100, 2), "%")
display(df["Default"].value_counts().rename("count").to_frame())

## 4. Data quality and exploratory analysis

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(25).to_frame("missing_percent"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Default", ax=axes[0])
axes[0].set_title("Fully paid vs charged-off loans")
axes[0].set_xticklabels(["Fully Paid (0)", "Charged Off (1)"])
if "loan_amnt" in df.columns:
    sns.boxplot(data=df, x="Default", y="loan_amnt", showfliers=False, ax=axes[1])
    axes[1].set_title("Loan amount by outcome")
plt.tight_layout()
plt.show()

In [ ]:
for col in ["purpose", "grade", "home_ownership", "term"]:
    if col in df.columns:
        rate = df.groupby(col, dropna=False)["Default"].agg(["mean", "count"]).sort_values("mean", ascending=False)
        display(rate.head(15).style.format({"mean": "{:.1%}"}))

## 5. Feature selection and leakage prevention
Use fields available at or near loan application/origination. Avoid payment outcomes, recovery amounts, last payment dates, collection data, and other information created after the loan was issued.

The candidate list below is intentionally conservative. Check the data dictionary and remove any field that is not known at the prediction point you define.

In [ ]:
candidate_features = [
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "addr_state",
    "dti", "delinq_2yrs", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc",
    "application_type"
]
features = [c for c in candidate_features if c in df.columns]
if not features:
    raise ValueError("None of the expected predictor columns were found. Inspect df.columns and update candidate_features.")

model_df = df[features + ["Default"]].copy()
print("Selected features:", features)
display(model_df.head())

## 6. Feature engineering
Create a few interpretable ratios only when their source fields exist. These are candidate indicators, not universal underwriting rules.

In [ ]:
if {"annual_inc", "loan_amnt"}.issubset(model_df.columns):
    annual_income = pd.to_numeric(model_df["annual_inc"], errors="coerce").replace(0, np.nan)
    loan_amount = pd.to_numeric(model_df["loan_amnt"], errors="coerce")
    model_df["Loan_to_Annual_Income"] = loan_amount / annual_income

if {"annual_inc", "installment"}.issubset(model_df.columns):
    annual_income = pd.to_numeric(model_df["annual_inc"], errors="coerce").replace(0, np.nan)
    installment = pd.to_numeric(model_df["installment"], errors="coerce")
    model_df["Annual_Installment_to_Income"] = (installment * 12) / annual_income

display(model_df.head())

## 7. Prepare training and test data
Split before fitting imputers, scalers, and encoders to prevent data leakage. Stratification preserves the default share.

In [ ]:
X = model_df.drop(columns="Default")
y = model_df["Default"].astype(int)

# Convert known numeric columns to numeric where possible.
for col in X.columns:
    if X[col].dtype == "object":
        # Keep categorical fields as categorical; engineered ratios remain numeric.
        pass

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 8. Train models
The class-weighted models help account for unequal class frequencies. Hyperparameters are starter values; use cross-validation and tuning for a stronger final submission.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1500, class_weight="balanced",
                                     random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=5,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])
}

if XGBOOST_AVAILABLE:
    # One-hot encoding is handled by the shared preprocessing pipeline.
    neg, pos = np.bincount(y_train.astype(int))
    models["XGBoost"] = Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85,
            scale_pos_weight=neg / max(pos, 1),
            eval_metric="logloss", random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)

## 9. Model comparison

In [ ]:
rows = []
model_probabilities = {}
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    model_probabilities[name] = proba
    rows.append({
        "Model": name,
        "Precision@0.5": precision_score(y_test, pred, zero_division=0),
        "Recall@0.5": recall_score(y_test, pred, zero_division=0),
        "F1@0.5": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "PR_AUC": average_precision_score(y_test, proba)
    })

comparison = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
display(comparison)

## 10. Select a model and review classification errors
This baseline uses the model with the highest ROC-AUC for demonstration. For an imbalanced dataset, also compare PR-AUC and choose a threshold based on the bank's risk appetite and review capacity.

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = models[best_name]
best_proba = model_probabilities[best_name]
best_pred = (best_proba >= 0.5).astype(int)

print("Selected baseline model:", best_name)
print(classification_report(y_test, best_pred, target_names=["Fully Paid", "Charged Off"], zero_division=0))

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted paid", "Predicted default"],
            yticklabels=["Actual paid", "Actual default"])
plt.title("Credit Default Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 11. Probability of Default (PD) and risk bands
The model score is a predicted probability, not automatically a calibrated or regulatory Probability of Default. The bands below are illustrative and should be validated and calibrated before real use.

In [ ]:
scored = X_test.copy()
scored["Actual_Default"] = y_test.values
scored["Predicted_PD"] = best_proba
scored["Predicted_Default"] = best_pred

scored["Risk_Band"] = pd.cut(
    scored["Predicted_PD"],
    bins=[-0.001, 0.10, 0.25, 0.50, 1.0],
    labels=["Low", "Moderate", "High", "Very High"]
)

risk_summary = scored.groupby("Risk_Band", observed=False).agg(
    Borrowers=("Actual_Default", "size"),
    Actual_Default_Rate=("Actual_Default", "mean"),
    Average_Predicted_PD=("Predicted_PD", "mean")
).reset_index()

display(risk_summary.style.format({
    "Actual_Default_Rate": "{:.1%}",
    "Average_Predicted_PD": "{:.1%}"
}))

## 12. Illustrative Expected Credit Loss (ECL)
Expected Credit Loss is commonly expressed using PD, LGD and EAD. This simplified calculation uses assumptions for LGD and EAD; it is not an IFRS 9-compliant ECL model. A production ECL framework needs approved definitions, forward-looking scenarios, staging, discounting and governance.

In [ ]:
# For this portfolio illustration, use loan amount as a simplified EAD proxy.
if "loan_amnt" in scored.columns:
    scored["EAD_Proxy"] = pd.to_numeric(scored["loan_amnt"], errors="coerce").fillna(0)
else:
    scored["EAD_Proxy"] = 0.0

LGD_ASSUMPTION = 0.45  # illustrative 45%; replace with validated recovery/LGD estimates
scored["LGD_Assumption"] = LGD_ASSUMPTION
scored["Expected_Loss_Illustrative"] = (
    scored["Predicted_PD"] * scored["LGD_Assumption"] * scored["EAD_Proxy"]
)

print("Illustrative expected loss on test sample:",
      f"{scored['Expected_Loss_Illustrative'].sum():,.2f}")
display(scored[["Predicted_PD", "Risk_Band", "EAD_Proxy",
                "LGD_Assumption", "Expected_Loss_Illustrative"]].head())

## 13. SQL practice using DuckDB (optional)
This section demonstrates SQL-style analysis on the prepared data. DuckDB may already be installed in Kaggle; if not, install it where internet access is enabled.

In [ ]:
try:
    import duckdb
    con = duckdb.connect()
    con.register("loan_data", model_df)
    sql_result = con.execute('''
        SELECT Default,
               COUNT(*) AS loan_count,
               AVG(Default) AS default_rate
        FROM loan_data
        GROUP BY Default
        ORDER BY Default
    ''').df()
    display(sql_result)
except Exception as e:
    print("DuckDB section skipped:", e)
    print("You can run equivalent SQL after loading the prepared table into a SQL database.")

## 14. Export results for Power BI
The scored test set is suitable for a portfolio dashboard prototype. Use anonymized records only and do not expose personal borrower information.

In [ ]:
scored.to_csv("/kaggle/working/credit_risk_scored_loans.csv", index=False)
comparison.to_csv("/kaggle/working/credit_risk_model_comparison.csv", index=False)
risk_summary.to_csv("/kaggle/working/credit_risk_band_summary.csv", index=False)
print("Saved exports in /kaggle/working/")

## 15. Power BI dashboard design
Recommended report pages:

1. **Portfolio Overview:** total loans, total exposure proxy, observed default rate, predicted risk distribution.
2. **Borrower Risk:** risk bands, grade, purpose, income and DTI analysis.
3. **Default Analysis:** default rate by grade, term, purpose and verification status.
4. **Model Performance:** ROC-AUC, PR-AUC, precision, recall and confusion matrix.
5. **Expected Loss:** illustrative PD × LGD × EAD by risk band.

## 16. Conclusion and limitations
After running the notebook, write down:
- Which model performed best and why.
- Which borrower characteristics were associated with higher default rates.
- How risk bands align with observed default rates.
- The illustrative expected-loss estimate and its assumptions.
- Limitations: historical lending data, selection bias, incomplete outcomes, dataset-specific definitions, fairness, calibration, drift and leakage risk.

### Suggested interview talking point
"I built a credit-default classification workflow using application-time borrower and loan features. I compared logistic regression, random forest and XGBoost, evaluated ranking and classification metrics, translated predicted scores into illustrative risk bands, and connected PD, LGD and EAD to an expected-loss view for Power BI reporting."